# Ungauged Return-Period Flood-Flow Model — Validation Report

Predicts LP3 flood quantiles **Q2–Q500** at any CONUS NHDPlus reach from catchment attributes, then applies the model CONUS-wide.

**Design.** Target = at-site LP3 quantiles (regenerated with the corrected peak-code mapping). Features = 55 NHDPlus/COMID *total-upstream* attributes (Wieczorek *Select Attributes*: drainage area, precip, PET, temperature, BFI, runoff, recharge, soils, land cover, dam storage). Trained on **1,443 unregulated, QC-passed gauges** (natural-flow model). Engine = **LightGBM** with an **index-flood decomposition** (predict log Q2 + non-negative consecutive log-increments) that guarantees monotone curves Q2 ≤ … ≤ Q500. Validation = **leave-HUC2-out** spatial cross-validation. Uncertainty = split-conformal bands calibrated on the out-of-region residuals.

## Skill by return period (leave-HUC2-out)

| return_period | n | log_r2 | log_nse | log_rmse | log_bias | cfs_kge | cfs_nse |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Q2 | 1443.000 | 0.819 | 0.819 | 0.231 | -0.012 | 0.642 | 0.695 |
| Q5 | 1443.000 | 0.802 | 0.802 | 0.236 | -0.016 | 0.654 | 0.678 |
| Q10 | 1443.000 | 0.787 | 0.787 | 0.244 | -0.018 | 0.645 | 0.651 |
| Q25 | 1443.000 | 0.762 | 0.762 | 0.259 | -0.019 | 0.615 | 0.593 |
| Q50 | 1443.000 | 0.740 | 0.740 | 0.273 | -0.019 | 0.577 | 0.537 |
| Q100 | 1443.000 | 0.714 | 0.714 | 0.289 | -0.019 | 0.528 | 0.470 |
| Q200 | 1443.000 | 0.687 | 0.687 | 0.306 | -0.020 | 0.467 | 0.399 |
| Q500 | 1443.000 | 0.647 | 0.647 | 0.331 | -0.020 | 0.375 | 0.305 |

log-R²/NSE in log₁₀ space; KGE/NSE(cfs) in real space. Skill decays monotonically toward rarer events, as expected.

![skill](fig_skill_by_rp.png)

![pred vs obs](fig_pred_vs_obs_q10.png)

## Baseline: USGS StreamStats regional regressions

| return_period | n_shared | model_log_r2 | baseline_log_r2 | model_log_nse | baseline_log_nse |
| --- | --- | --- | --- | --- | --- |
| Q2 | 569.000 | 0.830 | 0.887 | 0.830 | 0.887 |
| Q5 | 570.000 | 0.814 | 0.884 | 0.814 | 0.884 |
| Q10 | 570.000 | 0.795 | 0.877 | 0.795 | 0.877 |
| Q25 | 508.000 | 0.773 | 0.873 | 0.773 | 0.873 |
| Q50 | 539.000 | 0.747 | 0.855 | 0.747 | 0.855 |
| Q100 | 551.000 | 0.719 | 0.834 | 0.719 | 0.834 |
| Q200 | 491.000 | 0.659 | 0.782 | 0.659 | 0.782 |
| Q500 | 551.000 | 0.641 | 0.783 | 0.641 | 0.783 |

On the ~570 sites with published StreamStats regression estimates, the state-specific USGS equations still lead (e.g. Q10 log-R² ≈ 0.88 vs ≈ 0.80). The value here is a single, nationally consistent model with calibrated uncertainty and full CONUS coverage — including reaches StreamStats regression does not serve.

## Independent benchmark: NWM v3.0 retrospective

| RP | NWM R² | NWM bias (dex) | NWM med. ratio | ML R² | ML bias (dex) |
| --- | --- | --- | --- | --- | --- |
| Q2 | 0.15 | -0.32 | 0.57 | 0.82 | -0.01 |
| Q5 | 0.27 | -0.29 | 0.59 | 0.80 | -0.02 |
| Q10 | 0.32 | -0.28 | 0.60 | 0.79 | -0.02 |
| Q25 | 0.36 | -0.27 | 0.60 | 0.76 | -0.02 |
| Q50 | 0.35 | -0.27 | 0.60 | 0.74 | -0.02 |
| Q100 | 0.34 | -0.27 | 0.59 | 0.71 | -0.02 |
| Q200 | 0.30 | -0.27 | 0.59 | 0.69 | -0.02 |
| Q500 | 0.23 | -0.27 | 0.57 | 0.65 | -0.02 |

A second, fully independent check. From 45 years (1979–2023) of NWM v3.0 retrospective *daily* streamflow we take the water-year annual maxima and fit the same log-Pearson III, giving NWM-implied Q2–Q500 at each gauge reach (COMID join). Both estimators are scored out-of-sample against the at-site LP3 on the **1,440 unregulated QC reaches**: NWM is a physics model that never saw these peaks, and the ML column is its **leave-HUC2-out** prediction (not the in-sample fit).

The attribute-based ML model is the stronger ungauged estimator at every return period (e.g. Q10 R² ≈ 0.79 vs 0.32; Q100 ≈ 0.71 vs 0.34). NWM also runs **~40% low** (median ratio ≈ 0.57–0.60, bias ≈ −0.27 dex) — the expected consequence of annual maxima taken from *daily-mean* flow, which under-represents instantaneous peaks, compounded by the model's own error. NWM's skill peaks around Q25–Q50 and its spatial pattern corroborates the ML surface, but for flood-quantile *magnitude* at ungauged reaches the trained model is clearly preferable.

![nwm](fig_nwm_comparison.png)

## Spatial cross-validation residuals

![spatial](fig_spatial_cv_map.png)

Residuals are small on average but regionally structured (e.g. Gulf-coast under-prediction), consistent with prior findings — a target for future region-aware refinement.

## Feature importance (physical sanity)

![importance](fig_feature_importance.png)

The model leans on catchment size (stream length, drainage area), mean-annual precipitation, base-flow index and runoff — the expected physical drivers.

## Uncertainty

![interval](fig_interval_width.png)

90% conformal intervals; empirical out-of-fold coverage ≈ 0.90 by construction. Bands widen for rarer return periods.

## CONUS product

`ml/conus_predictions.parquet` — **2,691,344 COMIDs**, columns `q{rp}_cfs` / `q{rp}_lo_cfs` / `q{rp}_hi_cfs` for rp ∈ {{2,5,10,25,50,100,200,500}}, plus `TOT_BASIN_AREA` and `has_upstream_dam`.

**Caveats.** (1) The model is trained on unregulated gauges, so predictions are *natural* flood potential; reaches flagged `has_upstream_dam` should be read as such. (2) Most CONUS reaches are far smaller headwater catchments than the training gauges, so those predictions are extrapolation — the widened conformal bands express this but were calibrated at gauge scale.

**Mapping.** The table is delivered COMID-keyed (no geometry). To map it, join `COMID` to the **FEATUREID** field of any NHDPlusV2 catchment layer (or `COMID` of the flowlines) in QGIS/ArcGIS. The national NHDPlus catchment *raster* is not directly usable — its cells store GRIDCODE, whose crosswalk to COMID ships only in EPA's 7.8 GB Seamless Geodatabase.

In [ ]:
import pandas as pd
conus = pd.read_parquet('~/data/flood_hazard/ml/conus_predictions.parquet',
                        columns=['COMID','q10_cfs','q10_lo_cfs','q10_hi_cfs','TOT_BASIN_AREA'])
conus.describe()